# Feature screening — `price_dislocation`

Screening is **step 1** of feature analysis: confirm the feature is *computed correctly* (the **parity
check**) and that it *passes the hygiene gates* (the **signal is real**). Only features that clear
screening go on to the later steps (which time-scale per head, input shaping, out-of-sample across blocks).

Everything below is wiring — the machinery is shared, tested code:
- the **method** (the two heads, what each gate proves): [`METHOD.md`](METHOD.md);
- the **rules** for writing a feature (EMA types, inject/decay, the guard rails): [`AUTHORING.md`](../../src/boba/features/AUTHORING.md);
- the **engines** (`build_context`, `parity_check`, `run_gates`, …): `boba.research.screening`;
- the **feature** itself: `boba.features.price_dislocation`.

To screen a new feature: write its module under `src/boba/features/` (a vectorized builder + a streaming
class, per `AUTHORING.md`), register it, then copy this notebook and change §1 and the feature name.

## 1. The feature

`price_dislocation` measures how far a foreign source's price has drifted from the target's, in units
of the target's volatility. One leg per foreign source $s$:

$$
\text{price\_dislocation}_s \;=\;
\frac{\mathrm{EMA}_{\text{fast}}(g_s)\,-\,\mathrm{EMA}_{\text{slow}}(g_s)}{\sigma_{\text{ev}}}
\,,\qquad
g_s \;=\; \log m_s - \log m_{\text{target}}
$$

The gap $g_s$ is the log-price difference (foreign mid $m_s$ vs target mid $m_{\text{target}}$);
**fast − slow** smooths it at two spans on the trade clock ($n_{\text{fast}} < n_{\text{slow}}$), so the
difference isolates a *fresh* move; dividing by the volatility yardstick $\sigma_{\text{ev}}$ puts the
score in σ-units. A fan-out feature; `params = (n_fast, n_slow)`.

**Hypothesis** — a fresh gap predicts the target's catch-up; gaps tend to close.
**Disproof** — no predictive power at any time-scale, or power that vanishes once we control for the
volatility regime.

Full definition: `src/boba/features/price_dislocation.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
pl.Config.set_fmt_str_lengths(100)                           # show full gate-detail strings (polars truncates at 30 by default)
from boba.features import base
import boba.features.price_dislocation                      # registers the feature
from boba.research.screening import (build_context, parity_check, build_family,
                                     best_span, run_gates, echo_netted_ic, HeadConfig)

ctx  = build_context()                                       # Step 0: one block -> clock, yardsticks, targets, controls, raw stream
spec = base.get("price_dislocation")
print(f"block {ctx.block}")
print(f"{len(ctx.merged_ts):,} trade ticks   {len(ctx.anchor_ts):,} grid anchors   sources {ctx.sources}")

## 2. Is the calculation right? — the parity check

The streaming (production) build must reproduce the vectorized (analysis) build on real data to
floating-point round-off. The generic driver does this for any feature.

In [ ]:
PARITY_SPANS = [(1, 200), (10, 100)]                         # incl. n_fast=1 (the α=1 / no-smoothing leg)
rep = parity_check(ctx, spec, PARITY_SPANS)
print(rep)
assert rep.passed, "streaming build does not reproduce the vectorized build"

## 3. Is the signal real? — the gates

Two independent tests per head: **Gate A** (regime-invariant distribution) and **Gate B** (predicts over
the regime-invariant controls), plus a calm/mid/wild companion. We give the feature its in-sample best
span per head — span *selection* proper is step 2, not screening.

In [ ]:
GRID = [(nf, ns) for nf in (1, 10, 50, 200, 500, 1000)
                 for ns in (100, 500, 1000, 2000, 5000, 10000) if nf < ns]
family = build_family(ctx, spec.vectorized, GRID, n_jobs=18)

price_span = best_span(ctx, family, ctx.price_target)                          # signed feature -> σ_ev return
rate_span  = best_span(ctx, family, ctx.rate_target, score_magnitude=True)     # |feature| -> count
print(f"price-head span {price_span}    rate-head span {rate_span}")

gr_price = run_gates(family[price_span], ctx, HeadConfig.price(ctx))
gr_rate  = run_gates(family[rate_span],  ctx, HeadConfig.rate(ctx))
gr_price.to_polars()

In [ ]:
gr_rate.to_polars()

## 4. Real prediction, or an echo of the move already underway?

A causal feature can still fail to predict if its edge is the move already underway at the anchor. The
**echo-netted** forward IC controls for the trailing move — report that, not the raw IC.

In [ ]:
src = ctx.sources[0]                                         # one venue shown; each carries its own
e = echo_netted_ic(ctx, family[price_span][src])
vals = [e["raw"], e["backward"], e["netted"]]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["raw forward", "backward (echo)", "echo-netted"], vals, color=["C0", "0.6", "C2"])
ax.axhline(0, color="0.7", lw=0.8); ax.set_ylabel("rank-IC"); ax.set_title(f"echo-netting — {src}, price head")
for i, v in enumerate(vals):
    ax.text(i, v, f"{v:+.3f}", ha="center", va="bottom" if v >= 0 else "top")
fig.tight_layout(); plt.show()
print(f"raw {e['raw']:+.3f}   backward {e['backward']:+.3f}   echo-netted {e['netted']:+.3f}")

## Screening verdict

In [ ]:
passed = rep.passed and gr_price.passed and gr_rate.passed
print("SCREENING:", "PASS" if passed else "FAIL")
print(f"  parity (calc correct)   {'OK' if rep.passed else 'FAILED'}")
print(f"  price head (gates)      {'pass' if gr_price.passed else 'fail'}")
print(f"  rate head (gates)       {'pass' if gr_rate.passed else 'fail'}")

**If it passes**, `price_dislocation` is computed correctly and carries real signal for the head(s) above —
it earns a place in the rest of the process: which time-scale per head, input shaping, out-of-sample across
blocks, and the ship checklist. Screening is **not** ship-ready.